In [ ]:
import os
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor
import logging
import json
import numpy as np
from scipy.sparse import csr_array
from scipy.sparse.linalg import eigsh
import h5py
import matplotlib.pyplot as plt
import jax
import jax.numpy as jnp
from flax import nnx
from qiskit import transpile
from qiskit_ibm_runtime import QiskitRuntimeService, Sampler, RuntimeEncoder, RuntimeDecoder
from qiskit_addon_sqd.qubit import solve_qubit, sort_and_remove_duplicates, project_operator_to_subspace
from heavyhex_qft.triangular_z2 import TriangularZ2Lattice
from heavyhex_qft.plaquette_dual import PlaquetteDual
from skqd_z2lgt.circuits import make_step_circuits, compose_trotter_circuits
from skqd_z2lgt.ising_dmrg import ising_dmrg, get_mps_probs
from skqd_z2lgt.recovery_learning import preprocess
from skqd_z2lgt.crbm import ConditionalRBM
from skqd_z2lgt.train_crbm import cd_meanloss, make_l2_loss_fn, train_crbm
from skqd_z2lgt.sqd import qiskit_sqd

os.environ['CUDA_VISIBLE_DEVICES'] = '0,1,2,3,4,5,6,7'
os.environ['PATH'] = '/opt/julia/iiyama/julia/bin:' + os.environ['PATH']
jax.config.update('jax_enable_x64', True)

logging.basicConfig(level=logging.WARNING)
LOG = logging.getLogger('skqd_z2lgt')
LOG.setLevel(logging.INFO)

In [2]:
service = QiskitRuntimeService(instance='ICEPP-dedicated-temp-prem-us')
backend = service.backend('ibm_pittsburgh')

In [3]:
output_file = h5py.File('/data/iiyama/2dz2/full_workflow.h5', 'r+')

In [4]:
lattice = TriangularZ2Lattice('''
  ^ ^ ^
 * * * *
* * * * *
 * * * *
* * * * *
 * * * *
* * * * *
 * * * *
  v v v
''')
plaquette_energy = 0.8
delta_t = 0.5
num_steps = 5
basis_2q = 'cz'

dual_lattice = PlaquetteDual(lattice)
ising_hamiltonian = dual_lattice.make_hamiltonian(plaquette_energy)

num_lnk = lattice.num_links
num_vtx = lattice.num_vertices
num_plq = lattice.num_plaquettes

## Compute the approximate ground-state energy

In [5]:
try:
    dmrg_energy = output_file['dmrg_energy'][()]
    mps_states = output_file['mps_states'][()]
    mps_probs = output_file['mps_probs'][()]
except KeyError:
    compute = True
else:
    compute = False

if compute:
    for key in ['dmrg_energy', 'mps_states', 'mps_probs']:
        try:
            del output_file[key]
        except KeyError:
            pass
    dmrg_energy = ising_dmrg(ising_hamiltonian, filename='/tmp/ising_dmrg.h5')
    states, probs = get_mps_probs('/tmp/ising_dmrg.h5')
    output_file.create_dataset('dmrg_energy', data=dmrg_energy)
    output_file.create_dataset('mps_states', data=states)
    output_file.create_dataset('mps_probs', data=probs)

print('DMRG energy:', dmrg_energy)

After sweep 1 energy=-89.12827998746036  maxlinkdim=8 maxerr=2.45E-16 time=14.618
After sweep 2 energy=-89.14276232612958  maxlinkdim=13 maxerr=9.98E-11 time=0.059
After sweep 3 energy=-89.14276294517288  maxlinkdim=12 maxerr=9.19E-11 time=0.053
After sweep 4 energy=-89.1427629622154  maxlinkdim=11 maxerr=8.75E-11 time=0.049
After sweep 5 energy=-89.14276296227733  maxlinkdim=11 maxerr=8.75E-11 time=0.050


The latest version of Julia in the `release` channel is 1.11.7+0.x64.linux.gnu. You currently have `1.11.6+0.x64.linux.gnu` installed. Run:

  juliaup update

in your terminal shell to install Julia 1.11.7+0.x64.linux.gnu and update the `release` channel to that version.


DMRG energy: -89.14276296227733


In [6]:
try:
    plaq_data = output_file['plaq_data'][()]
except KeyError:
    plaq_data = None

## Compose the circuits and run the experiments

In [7]:
if plaq_data is None:
    try:
        job = service.job(output_file['job_id'][()].decode())
    except KeyError:
        job = None

    if job is None:
        try:
            del output_file['job_id']
        except KeyError:
            pass

        layout = lattice.layout_heavy_hex(backend.coupling_map, backend_properties=backend.properties(), basis_2q=basis_2q)
        full_step, fwd_step, bkd_step, measure = transpile(
            make_step_circuits(lattice, plaquette_energy, delta_t, basis_2q),
            backend=backend, initial_layout=layout, optimization_level=2
        )
        id_step = fwd_step.compose(bkd_step)
        exp_circuits = compose_trotter_circuits(full_step, measure, num_steps)
        ref_circuits = compose_trotter_circuits(id_step, measure, num_steps)

        sampler = Sampler(backend)
        job = sampler.run(exp_circuits + ref_circuits, shots=backend.configuration().max_shots)
        output_file.create_dataset('job_id', data=job.job_id())

    result = job.result()

## Convert link-state counts to plaquette-state arrays with MWPM

In [8]:
if plaq_data is None:
    try:
        del output_file['plaq_data']
    except KeyError:
        pass

    with ProcessPoolExecutor() as executor:
        futures = []
        for res in result:
            futures.append(executor.submit(preprocess, res.data.c.get_counts(), dual_lattice))
        step_plaq_data = [future.result() for future in futures]

    dataset = output_file.create_dataset('plaq_data',
                                         shape=(2 * num_steps,) + step_plaq_data[0].shape,
                                         dtype=step_plaq_data[0].dtype)
    for istep, data in enumerate(step_plaq_data):
        dataset[istep] = data

    plaq_data = dataset[()]

exp_data = plaq_data[:num_steps]
exp_data_vtx = exp_data[:, :, :num_vtx]
exp_data_plq = exp_data[:, :, num_vtx:]
ref_data = plaq_data[num_steps:]
ref_data_vtx = ref_data[:, :, :num_vtx]
ref_data_plq = ref_data[:, :, num_vtx:]

## Test the SKQD accuracy without configuration recovery

In [9]:
try:
    bitstring_matrix_raw = output_file['skqd_raw/bitstring_matrix'][()]
    data = output_file['skqd_raw/ham_proj/data'][()]
    indices = output_file['skqd_raw/ham_proj/indices'][()]
    indptr = output_file['skqd_raw/ham_proj/indptr'][()]
    shape = (indptr.shape[0] - 1,) * 2
    ham_proj_raw = csr_array((data, indices, indptr), shape=shape)
    energy_raw = output_file['skqd_raw/energy'][()]
except KeyError:
    compute = True
else:
    compute = False

if compute:
    try:
        del output_file[f'skqd_raw']
    except KeyError:
        pass

    bitstrings = exp_data_plq.reshape(-1, num_plq)
    bitstring_matrix_raw, ham_proj_raw, energy_raw, _ = qiskit_sqd(bitstrings, ising_hamiltonian, jax_device_id=-1)

    group = output_file.create_group('skqd_raw')
    group.create_dataset('bitstring_matrix', data=bitstring_matrix_raw)
    group.create_dataset('energy', data=energy_raw)
    subgroup = group.create_group('ham_proj')
    subgroup.create_dataset('data', data=ham_proj_raw.data)
    subgroup.create_dataset('indices', data=ham_proj_raw.indices)
    subgroup.create_dataset('indptr', data=ham_proj_raw.indptr)

LOG.info('Basis size: %d', bitstring_matrix_raw.shape[0])
LOG.info('Matrix density %d', ham_proj_raw.data.shape[0])
LOG.info('Raw bitstrings energy: %f', energy_raw)

INFO:skqd_z2lgt:Basis size: 493956
INFO:skqd_z2lgt:Matrix density 544508
INFO:skqd_z2lgt:Raw bitstrings energy: -87.980865


## Test SKQD with random plaquette flips

In [12]:
def skqd_random(mean_activation, num_gen, seed, jax_device_id):
    rng = np.random.default_rng(seed)
    uniform = rng.random((num_steps, ref_data.shape[1], num_gen, num_plq))
    flips = np.asarray(uniform < mean_activation[:, None, None, :], dtype=np.uint8)
    bitstrings = np.concatenate([
        exp_data_plq.reshape((-1, num_plq)),
        (exp_data_plq[:, :, None, :] ^ flips).reshape((-1, num_plq))
    ], axis=0)
    return qiskit_sqd(bitstrings, ising_hamiltonian, jax_device_id=jax_device_id)


In [13]:
num_exps = jax.device_count()
num_gen = 10

mean_activation = np.mean(ref_data_plq, axis=1)

bitstring_matrix_rnd = {}
ham_proj_rnd = {}
energy_rnd = {}

futures = {}

with ThreadPoolExecutor() as executor:
    for iexp in range(num_exps):
        try:
            bitstring_matrix_rnd[iexp] = output_file[f'skqd_rnd_{iexp}/bitstring_matrix']
            data = output_file[f'skqd_rnd_{iexp}/ham_proj/data'][()]
            indices = output_file[f'skqd_rnd_{iexp}/ham_proj/indices'][()]
            indptr = output_file[f'skqd_rnd_{iexp}/ham_proj/indptr'][()]
            shape = (indptr.shape[0] - 1,) * 2
            ham_proj_rnd[iexp] = csr_array((data, indices, indptr), shape=shape)
            energy_rnd[iexp] = output_file[f'skqd_rnd_{iexp}/energy'][()]
        except KeyError:
            futures[iexp] = executor.submit(skqd_random, mean_activation, num_gen, 12345 + iexp, iexp)

    for iexp, future in futures.items():
        try:
            del output_file[f'skqd_rnd_{iexp}']
        except KeyError:
            pass

        result = future.result()
        bitstring_matrix_rnd[iexp] = result[0]
        ham_proj_rnd[iexp] = result[1]
        energy_rnd[iexp] = result[2]

        group = output_file.create_group(f'skqd_rnd_{iexp}')
        group.create_dataset('bitstring_matrix', data=bitstring_matrix_rnd[iexp])
        group.create_dataset('energy', data=energy_rnd[iexp])
        subgroup = group.create_group('ham_proj')
        subgroup.create_dataset('data', data=ham_proj_rnd[iexp].data)
        subgroup.create_dataset('indices', data=ham_proj_rnd[iexp].indices)
        subgroup.create_dataset('indptr', data=ham_proj_rnd[iexp].indptr)

for iexp in range(num_exps):
    LOG.info('Experiment %d basis size: %d', iexp, bitstring_matrix_rnd[iexp].shape[0])
    LOG.info('Matrix density %d', ham_proj_rnd[iexp].data.shape[0])
    LOG.info('Energy %f', energy_rnd[iexp])

INFO:skqd_z2lgt:Experiment 0 basis size: 5492549
INFO:skqd_z2lgt:Matrix density 5351361
INFO:skqd_z2lgt:Energy -88.063754
INFO:skqd_z2lgt:Experiment 1 basis size: 5492548
INFO:skqd_z2lgt:Matrix density 5351357
INFO:skqd_z2lgt:Energy -88.077726
INFO:skqd_z2lgt:Experiment 2 basis size: 5492537
INFO:skqd_z2lgt:Matrix density 5352344
INFO:skqd_z2lgt:Energy -88.078204
INFO:skqd_z2lgt:Experiment 3 basis size: 5492466
INFO:skqd_z2lgt:Matrix density 5351715
INFO:skqd_z2lgt:Energy -88.057837
INFO:skqd_z2lgt:Experiment 4 basis size: 5492529
INFO:skqd_z2lgt:Matrix density 5350730
INFO:skqd_z2lgt:Energy -88.044288
INFO:skqd_z2lgt:Experiment 5 basis size: 5492526
INFO:skqd_z2lgt:Matrix density 5351332
INFO:skqd_z2lgt:Energy -88.080117
INFO:skqd_z2lgt:Experiment 6 basis size: 5492492
INFO:skqd_z2lgt:Matrix density 5351803
INFO:skqd_z2lgt:Energy -88.059125
INFO:skqd_z2lgt:Experiment 7 basis size: 5492598
INFO:skqd_z2lgt:Matrix density 5351222
INFO:skqd_z2lgt:Energy -88.073282


## Train CRBMs for configuration recovery

In [14]:
num_h = 256
num_epochs = 10
batch_size = 32
lr = 0.001
l2w_weights = 1.
l2w_biases = 0.2

filename_template = f'/data/iiyama/2dz2/crbm_models/p{num_plq}_i%d_h{num_h}_b{batch_size}_lr{lr:.0e}_e{num_epochs}_cdm_l2w{l2w_weights:.0e}_l2b{l2w_biases}.h5'

def make_model(istep):
    model_filename = filename_template % istep
    if os.path.exists(model_filename):
        model = ConditionalRBM.load(model_filename)
    else:
        train_v = ref_data_plq[istep, :80_000]

        mean_activation = np.mean(train_v, axis=0)
        mean_activation = np.where(np.isclose(mean_activation, 0.), 1.e-6, mean_activation)
        bias_init = np.log(mean_activation / (1. - mean_activation))
        h_sparsity = 0.01

        with jax.default_device(jax.devices()[istep % (jax.device_count() - 1)]):
            rngs = nnx.Rngs(params=0, sample=1)
            model = ConditionalRBM(num_vtx, num_plq, num_h, rngs=rngs)
            model.bias_v.value = bias_init
            model.bias_h.value = jnp.full(num_h, np.log(h_sparsity / (1. - h_sparsity)))

    return model

def train_model(istep, model, num_epochs, records=None):
    model_filename = filename_template % istep
    if os.path.exists(model_filename) and records is None:
        with h5py.File(model_filename.replace('.h5', '_records.h5'), 'r') as source:
            group = source['records']
            records = {key: array[()] for key, array in group.items()}

    else:
        LOG.info('Start training model for step %d', istep)
        train_dataset = ref_data[istep, :80_000]
        test_dataset = ref_data[istep, 80_000:]

        with jax.default_device(jax.devices()[istep % (jax.device_count() - 1)]):
            loss_fn = make_l2_loss_fn(cd_meanloss, l2w_weights, l2w_biases)
            records = train_crbm(model, train_dataset, test_dataset, batch_size, num_epochs,
                                 loss_fn, lr, records=records)

        model.save(model_filename)
        with h5py.File(model_filename.replace('.h5', '_records.h5'), 'w') as out:
            group = out.create_group('records')
            for key, array in records.items():
                group.create_dataset(key, data=array)

    return records

In [15]:
futures = {}
models = {}
records = {}

with ThreadPoolExecutor() as executor:
    for istep in range(num_steps):
        models[istep] = model = make_model(istep)
        futures[istep] = executor.submit(train_model, istep, model, 20)

    for istep, future in futures.items():
        records[istep] = future.result()

## Test SKQD with configuration recovery

In [16]:
def generate_data(model, cond_bits, base_bits, num_gen, batch_size):
    num_batches = int(np.ceil(cond_bits.shape[0] / batch_size).astype(int))
    gen_data = np.empty((base_bits.shape[0], num_gen, base_bits.shape[1]), dtype=base_bits.dtype)
    for ibatch in range(num_batches):
        start = ibatch * batch_size
        end = (ibatch + 1) * batch_size
        flips = model.sample(cond_bits[start:end], size=num_gen).transpose((1, 0, 2))
        gen_data[start:end] = base_bits[start:end, None, :] ^ flips

    return gen_data.reshape((-1, base_bits.shape[1]))

In [17]:
try:
    bitstring_matrix_rcv = output_file['skqd_rcv/bitstring_matrix'][()]
    data = output_file['skqd_rcv/ham_proj/data'][()]
    indices = output_file['skqd_rcv/ham_proj/indices'][()]
    indptr = output_file['skqd_rcv/ham_proj/indptr'][()]
    shape = (indptr.shape[0] - 1,) * 2
    ham_proj_rcv = csr_array((data, indices, indptr), shape=shape)
    energy_rcv = output_file['skqd_rcv/energy'][()]
except KeyError:
    compute = True
else:
    compute = False

if compute:
    try:
        del output_file['skqd_rcv']
    except KeyError:
        pass

    num_gen = 10
    batch_size = 10000
    gen_data = np.empty((num_steps, exp_data_plq.shape[1] * num_gen, num_plq), dtype=plaq_data.dtype)

    with ThreadPoolExecutor() as executor:
        for istep in range(num_steps):
            futures[istep] = executor.submit(generate_data, models[istep], exp_data_vtx[istep],
                                             exp_data_plq[istep], num_gen, batch_size)

        for istep, future in futures.items():
            gen_data[istep] = future.result()

    bitstrings = np.concatenate([
        exp_data_plq.reshape((-1, num_plq)),
        gen_data.reshape((-1, num_plq))
    ], axis=0)
    bitstring_matrix_rcv, ham_proj_rcv, energy_rcv, _ = qiskit_sqd(bitstrings, ising_hamiltonian, jax_device_id=-1)

    group = output_file.create_group('skqd_rcv')
    group.create_dataset('bitstring_matrix', data=bitstring_matrix_rcv)
    group.create_dataset('energy', data=energy_rcv)
    subgroup = group.create_group('ham_proj')
    subgroup.create_dataset('data', data=ham_proj_rcv.data)
    subgroup.create_dataset('indices', data=ham_proj_rcv.indices)
    subgroup.create_dataset('indptr', data=ham_proj_rcv.indptr)

LOG.info('Basis size: %d', bitstring_matrix_rcv.shape[0])
LOG.info('Matrix density %d', ham_proj_rcv.data.shape[0])
LOG.info('Recovered bitstrings energy: %f', energy_rcv)

INFO:skqd_z2lgt:Basis size: 5484215
INFO:skqd_z2lgt:Matrix density 5545029
INFO:skqd_z2lgt:Recovered bitstrings energy: -88.271485


In [18]:
dmrg_energy, energy_raw, energy_rnd, energy_rcv

(np.float64(-89.14276296227733),
 np.float64(-87.98086532493896),
 {0: np.float64(-88.06375433197425),
  1: np.float64(-88.0777263622731),
  2: np.float64(-88.0782043187388),
  3: np.float64(-88.05783708544328),
  4: np.float64(-88.04428762357988),
  5: np.float64(-88.08011696391956),
  6: np.float64(-88.059124602007),
  7: np.float64(-88.0732819237293)},
 np.float64(-88.27148466247175))